[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/auteurs/cross_entropy_colab.ipynb)

**Notebook des auteurs** (Guth, Kadkhodaie, Simoncelli, 2025), adapté pour Colab et pour un ordinateur sans GPU par le TP « score matching dual » : les cellules marquées « TP » sont des ajouts ; le reste est inchangé.

Réglages des auteurs : 150 000 évaluations du réseau couleur (GPU indispensable). Par défaut : 2 000 images d'entraînement et 2 000 de test.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter les cellules dans l'ordre. Réglages réduits par défaut, même avec un GPU ; réglages des auteurs (non vérifié) : `if False:` -> `if True:` dans la cellule de compatibilité ; données : même changement dans la cellule « Données » (téléchargement de 1 Go).

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_color_lr0.0005_1Msteps10decays/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, ""))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, ""))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

In [ ]:
# Compatibilité (ajout du TP) : le code des auteurs a été écrit pour un GPU et une version de PyTorch antérieure
# à 2.6 (torch.load lisait alors par défaut des objets Python, et non seulement des poids).
import functools, sys, torch
sys.argv = sys.argv[:1]   # main.parse_args lit la ligne de commande : celle du noyau Jupyter n'est pas la sienne
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not getattr(torch.load, "_tp", False):
    _torch_load = torch.load
    def _load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)               # points de contrôle des auteurs (dépôt de confiance)
        if not torch.cuda.is_available():
            kwargs.setdefault("map_location", "cpu")           # tenseurs enregistrés sur GPU
        return _torch_load(*args, **kwargs)
    _load._tp = True
    torch.load = _load
import networks.model as _nm
if not torch.cuda.is_available() and not getattr(_nm.Model.__init__, "_tp", False):
    # sans GPU, Model n'enveloppe pas le réseau dans DataParallel, alors que main.py lit model.network.module
    _init = _nm.Model.__init__
    def _init_cpu(self, network, *args, **kwargs):
        _init(self, network, *args, **kwargs)
        self.network = torch.nn.DataParallel(self.network)    # sans GPU, DataParallel appelle le réseau tel quel
    _init_cpu._tp = True
    _nm.Model.__init__ = _init_cpu
os.makedirs("tp/resultats/auteurs", exist_ok=True)   # écritures du notebook (au lieu de outputs/, laissé intact)
# Réglages : réduits par défaut, même avec un GPU (le calcul se fait alors sur le GPU) ; réglages des auteurs (non
# vérifié, GPU indispensable) : remplacer False par True ci-dessous.
REGLAGES_AUTEURS = False
if False:
    REGLAGES_AUTEURS = torch.cuda.is_available()
print("calcul sur", DEVICE, "- réglages", "des auteurs" if REGLAGES_AUTEURS else "réduits")

In [ ]:
# Données (ajout du TP ; une fois par machine, sur Colab à chaque nouvelle session) : ImageNet64 n'est pas distribué
# par les auteurs. On le reconstruit au format attendu par data.py (version de Chrabaszcz et al., 2017) : validation
# complète (fichier officiel de image-net.org, 50 000 images, 534 Mo) et 58 235 premières images d'entraînement. Rien
# n'est téléchargé par défaut : remplacer False par True ci-dessous.
if False:
    import sys
    sys.path.insert(0, "tp")
    import tp_outils
    if True:
        # images d'entraînement depuis le miroir HuggingFace ChocolateDave/imagenet-64 (version figée, 483 Mo)
        tp_outils.preparer_imagenet64_auteurs(entrainement="chocolatedave")
    else:
        # mêmes images depuis image-net.org (début du premier fichier d'entraînement officiel : 1,37 Go)
        tp_outils.preparer_imagenet64_auteurs(entrainement="imagenet")

# Computing cross-entropy for several models

In [ ]:

import numpy as np
from matplotlib import pyplot as plt
import torch

from tqdm.auto import tqdm, trange

from main import *

params = {
      'text.usetex': False,  # TP : pas de LaTeX sur Colab
      'font.family': 'sans serif'
}
matplotlib.rcParams.update(params)

## Load energy models (on ImageNet color)

In [ ]:
def load_exp(name, step="last", log=True, dataloaders=False):
    """ Load an experiment with a given name. step can be an integer, "best", or "last" (default). """
    exp_dir = Path("models") / name

    with open(exp_dir / "args.json") as f:
        args_dict = json.load(f)
    args_dict["dir"] = "models"  # TP : args.json indique "experiments"

    ctx = TrainingContext(**args_dict, step=step, key_remap=None, seed=None, dataloaders=dataloaders, writer=False)
    if log:
        print(f"{name}: retrieved model at step {ctx.step} and test loss {ctx.test_perf.loss:.2e}")

    # Disable DataParallel (needed for Hessian computation)
    ctx.model.network = ctx.model.network.module

    # Put in eval mode and disable gradients with respect to all parameters.
    ctx.model.eval()
    for p in ctx.model.parameters():
        p.requires_grad = False

    return ctx

ctxs = {
    "color": load_exp("finalclean_imagenet64_color_lr0.0005_1Msteps10decays", dataloaders=True),
    # Add other models here as needed.
}
default_ctx = ctxs["color"]
dataset_info = default_ctx.dataset_info
d = dataset_info.dimension

## Compute cross-entropy/NLL

In [ ]:
# Compare their probabilities on clean images

def samples_energy(dataloader, t=0):
    default_ctx.time_tracker.reset()

    xs = []
    es = defaultdict(list)
    for x in tqdm(dataloader):
        x = x[0]
        xs.append(x.cpu())
        clean = x.to(DEVICE)
        input = model_input(clean, noise_level=t)

        default_ctx.time_tracker.switch("forward")
        for k, c in ctxs.items():
            output = c.model(input, compute_scores=False, create_graph=False)  # (B,)
            es[k].append(output.energy.cpu())

    print(default_ctx.time_tracker.pretty_print())

    xs = torch.cat(xs, dim=0)
    es = {k: torch.cat(e, dim=0) for k, e in es.items()}
    return xs, es

# Compute energies in nats (summed over dimensions) over the whole dataset.
train_dataloader = default_ctx.new_dataloader(train=True, batch_size=2048, num_samples=50_000 if REGLAGES_AUTEURS else 2_000)  # We don't need the full dataset here.
imgs_train, energies_train = samples_energy(train_dataloader)
test_dataloader = default_ctx.new_dataloader(train=False, batch_size=2048, data_subset=None if REGLAGES_AUTEURS else slice(0, 2_000))
imgs_test, energies_test = samples_energy(test_dataloader)

# Compute normalization constant by computing average energy at large noise level
t = 1e3
test_dataloader = default_ctx.new_dataloader(train=False, batch_size=2048, data_subset=None if REGLAGES_AUTEURS else slice(0, 2_000))
_, energies_t = samples_energy(test_dataloader, t=t)
constants = {key: d/2 * np.log(2 * np.pi * np.e * t) - energy.mean() for key, energy in energies_t.items()}

for key in ctxs:
    nll_train = 8 + (energies_train[key].mean() + constants[key]) / (d * np.log(2))
    nll_test = 8 + (energies_test[key].mean() + constants[key]) / (d * np.log(2))
    print(f"{key}: train NLL {nll_train:.2f}, test NLL {nll_test:.2f}")

## Releasing GPU memory

In [ ]:
1/0

ZeroDivisionError: division by zero

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()